# TinyCeNN-LM — Fast Story + Anti-Repetition Training

This notebook specializes the compact Transformer-free 8-shard Top-2 TinyCeNN model for **short coherent stories**. It deliberately skips the slow held-out benchmark.

Changes aimed at the repetition problem:
- train on `roneneldan/TinyStories`;
- CE + recent-token **unlikelihood loss**;
- lower router balancing pressure so shards can specialize;
- story decoding with repetition penalty + 4-gram blocking + top-p sampling;
- 45-minute hard training cap.


In [ ]:
import subprocess, sys, pathlib, importlib
subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), 'huggingface_hub'], check=True)
SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
import tinycenn_lm
print('TinyCeNN import: PASS', tinycenn_lm.__file__)


## Hugging Face login
Store a **write** token in Colab Secrets as `HF_TOKEN`. The token is never written into the notebook.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets first.')
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()['name']
print('HF user:', HF_USER)


## Settings
No validation/evaluation dataset is loaded in this workflow.


In [ ]:
SOURCE_REPO = 'vtava/TinyCeNN-LM-Sharded-MoE-Top2'
TARGET_REPO = f'{HF_USER}/TinyCeNN-LM-Story-AntiRepeat'
MAX_TOKENS = 20_000_000
MAX_RUNTIME_MINUTES = 45
LEARNING_RATE = 1e-4
REPEAT_WEIGHT = 0.20
REPEAT_WINDOW = 32
OUTPUT_DIR = pathlib.Path('/content/TinyCeNN-LM/checkpoints/story-antirepeat')
print('source:', SOURCE_REPO)
print('target:', TARGET_REPO)


## Download the current compact sharded model


In [ ]:
SOURCE_DIR = pathlib.Path(snapshot_download(SOURCE_REPO, token=HF_TOKEN))
print('source checkpoint:', SOURCE_DIR)
required = ['sharded_moe_cenn_student.pt', 'sharded_moe_student_config.json']
for name in required:
    if not (SOURCE_DIR / name).exists():
        raise FileNotFoundError(f'{SOURCE_REPO} is missing {name}')


## Train — no slow evaluation


In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / 'scripts/train_story_antirepeat.py'),
    '--source-dir', str(SOURCE_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--max-tokens', str(MAX_TOKENS),
    '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
    '--learning-rate', str(LEARNING_RATE),
    '--repeat-weight', str(REPEAT_WEIGHT),
    '--repeat-window', str(REPEAT_WINDOW),
    '--router-aux-weight', '0.0005',
    '--router-z-weight', '0.0001',
    '--log-every', '20',
    '--save-every', '1000',
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## Training report


In [ ]:
import json
report = json.loads((OUTPUT_DIR / 'story_training_report.json').read_text())
print(json.dumps(report, indent=2))
assert report['evaluation_performed'] is False


## Publish the story checkpoint


In [ ]:
model_card = f'''---
language: en
license: mit
tags:
- text-generation
- cenn
- mixture-of-experts
- tinystories
---
# TinyCeNN-LM Story AntiRepeat

Transformer-free TinyCeNN story specialization derived from `{SOURCE_REPO}`.

Training uses TinyStories with causal CE plus recent-token unlikelihood loss. No held-out benchmark is run in this fast workflow.

Recommended decoding: temperature 0.78, top-p 0.90, top-k 40, repetition penalty 1.18, no-repeat 4-gram.
'''
(OUTPUT_DIR / 'README.md').write_text(model_card)
api.create_repo(TARGET_REPO, repo_type='model', exist_ok=True)
api.upload_folder(repo_id=TARGET_REPO, repo_type='model', folder_path=str(OUTPUT_DIR), commit_message='Publish TinyCeNN story anti-repetition model')
print('Published:', f'https://huggingface.co/{TARGET_REPO}')


## Reload and test short stories immediately
This is only a generation test, not a dataset evaluation.


In [ ]:
import torch
from transformers import AutoTokenizer
from tinycenn_lm.sharded_moe import ShardedMoECeNNReplacementLayer, build_sharded_moe_student
from tinycenn_lm.story import repeated_ngram_fraction, story_generation_kwargs
REMOTE_DIR = pathlib.Path(snapshot_download(TARGET_REPO, token=HF_TOKEN, force_download=False))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type == 'cuda' else torch.float32)
tokenizer = AutoTokenizer.from_pretrained(REMOTE_DIR, use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
model = build_sharded_moe_student(REMOTE_DIR, device=device, dtype=dtype).eval()
assert not any('self_attn' in name for name, _ in model.named_modules())
assert any(isinstance(m, ShardedMoECeNNReplacementLayer) for m in model.modules())
print('Transformer-free story model reload: PASS')
prompts = [
    'Story:\nMia found a small robot under a tree in the park.',
    'Story:\nA little fox wanted to reach the top of a snowy mountain.',
    'Story:\nTom had a red ball, but one rainy morning it disappeared from the garden.',
]
gen_kwargs = story_generation_kwargs(tokenizer, max_new_tokens=110)
torch.manual_seed(7)
for prompt in prompts:
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    with torch.inference_mode():
        out = model.generate(ids, **gen_kwargs)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    continuation = text[len(prompt):].strip() if text.startswith(prompt) else text
    print('\n' + '=' * 80)
    print(text)
    print('repeated 3-gram fraction:', f'{repeated_ngram_fraction(continuation, 3):.2%}')
